# MTH3302 : Méthodes probabilistes et statistiques pour l'I.A.

Anne Martin<br/>
Polytechnique Montréal<br/>


---
# Projet H2026 : Prédiction de la quantité de particules fines dans l'air de Montréal

### Contexte
La Ville de Montréal mesure la qualité de l’air via l'Indice de la Qualité de l'Air (IQA). Un enjeu majeur de santé publique concerne les particules fines (PM), des aérosols en suspension capables de pénétrer profondément dans le système respiratoire. Un dépassement de la limite acceptable (fixée à 50 par la Ville) représente un risque potentiel pour la santé, en particulier pour les populations vulnérables (enfants, aînés, asthmatiques).

L'évolution de ces particules est complexe : elle dépend des émissions polluantes urbaines, mais aussi de variables météorologiques qui favorisent ou empêchent la dispersion des polluants dans l'atmosphère.

### Objectif
L'objectif de ce projet est de prédire, pour sept stations de mesure réparties sur l'île de Montréal, l'IQA des particules fines (PM) pour chaque jour de l'année 2025.

### Données
Pour construire et valider nos modèles, nous exploitons les quatre fichiers de données suivants (2007-2025) :

* **`qualite-de-lair_train.csv`** & **`qualite-de-lair_test.csv`** : Historique et cibles de l'IQA pour les particules fines ($PM$), l'ozone ($O_3$), le dioxyde d'azote ($NO_2$) et le dioxyde de soufre ($SO_2$) pour les stations :
    * *Saint-Joseph, Saint-Jean-Baptiste, Aéroport de Montréal, Caserne 17, York/Roberval, Saint-Dominique et Anjou.*
* **`meteo_train.csv`** & **`meteo_test.csv`** : Observations météorologiques quotidiennes de la station de l'aéroport Montréal-Trudeau :
    * **Précipitations** : Pluie, neige, précipitations totales et neige au sol.
    * **Conditions atmosphériques** : Température, humidité relative, point de rosée et pression.
    * **Vent et Visibilité** : Vitesse du vent et visibilité (min, max et moyennes).


---

In [ ]:
using CSV, DataFrames, Dates, Gadfly, GLM, Statistics, Plots, Random, LinearAlgebra, StatsModels, Distributions

---
## 1. Chargement de données
#### Données de la qualité de l'air


In [ ]:
train_air = CSV.read("data/qualite-de-lair_train.csv", DataFrame)
first(train_air, 5)

##### Données météo de la station de l'aéroport de Montréal

In [ ]:
train_meteo = CSV.read("data/meteo_train.csv", DataFrame)

Nous fusionnons les deux fichiers d'entraînement par la date.

Avec innerjoin, on garantit que toutes les lignes de train ont un stationId valide et une variable réponse PM interprétable.

In [ ]:
train = innerjoin(train_air, train_meteo, on=:Date, makeunique=true)
println("Dimensions du jeu d'entraînement fusionné : ", size(train))
first(train, 5)

Afin de faire des observations sur les données et en tirer des conclusions pertinentes, nous avons écrit une fonction qui génère les $R^2$ de régression linéaires simples.

In [ ]:
function safe_r2_simple_regression(x, y)
    valid_idx = .!ismissing.(x) .& .!ismissing.(y)

    if sum(valid_idx) < 10
        return missing
    end

    x_clean = Float64.(x[valid_idx])
    y_clean = Float64.(y[valid_idx])

    x̄ = mean(x_clean)
    ȳ = mean(y_clean)

    sxx = sum((x_clean .- x̄).^2)
    if sxx == 0
        return missing
    end

    sxy = sum((x_clean .- x̄) .* (y_clean .- ȳ))

    β1 = sxy / sxx
    β0 = ȳ - β1 * x̄

    y_hat = β0 .+ β1 .* x_clean

    sse = sum((y_clean .- y_hat).^2)
    sst = sum((y_clean .- ȳ).^2)

    if sst == 0
        return missing
    end

    r2 = 1 - sse / sst
    return (β0, β1, r2)
end

---
## 2. Analyse exploratoire


Afin de se familiariser avec les données, nous avons effectué une analyse sur les différentes variables et les différents résultats présents dans les données. Nous allons aussi faire des graphiques pour visualiser les données et les corrélations possibles entre les différentes variables.

#### 2.1 Tendance générale
Commençons par afficher l'IQA de PM moyenne par année dans toutes les stations, de quoi avec une vision générale notre variable d'intérêt.

In [ ]:
set_default_plot_size(16cm, 10cm)
df = dropmissing(train_air)
df.Year = year.(df.Date)
df_average_per_year = combine(groupby(df, :Year), :PM => mean => :PM)
general_tendance = Gadfly.plot(df_average_per_year, x=:Year, y=:PM, Geom.line, Geom.point, Guide.title("Tendance générale de l'IQA de PM par année"))

Nous pouvons aperçevoir une tendance à la baisse quand à l'IQA de PM, semblant suivre une lois Inverse Gamma. Cependant, nous pouvons aperçevoir une remonté à partir de l'année 2020, puis une nouvelle chute apparait dès 2023 et se continuant tout au long de l'année 2024, où elle atteint son plus bas point depuis 17 ans.

Nous sommes conscients que des données abérrantes pourraient fausser des moyennes, ce qui pourrait rendre les données du précédent graphique innexactes. Nous aimerions vérifier si certaines données pourraient être considérés comme abérrantes visuellement de par le graphique suivant.

In [ ]:
set_default_plot_size(30cm, 10cm)
df = dropmissing(train_air)
general_tendance = Gadfly.plot(df, x=:Date, y=:PM, Geom.line, Geom.point, Guide.title("Tendance générale de l'IQA de PM par année"))

Nous constatons que certaines données semblent diverger de la tendance générale. Ces données pourraient constituer des données soit extrêmes ou aberrantes, nous allons devoir les traiter et décider lors de la création de nos modèles soit de les garder ou de s'en débarrasser afin de conserver une consistance dans notre jeu de données.

#### 2.2 Tendance des stations individuelles

Afin de voir l'évolution de la quantité de particules fines au cours du temps, nous avons créé des graphiques qui représentent l'évolution de la quantité de particules fines au cours du temps pour chaque station. Ceci nous permettra de voir si les tendances temporelles sont similaires ou différentes selon les stations, et si certaines stations présentent des pics ou des baisses marquées à certains moments.

In [ ]:
stations = unique(train_air.nom)
plots = []

for station in stations
    df = dropmissing(train_air[train_air.nom .== station, [:Date, :PM]])

    df.Year = year.(df.Date)

    df_yearly = combine(groupby(df, :Year),
        :PM => mean => :PM
    )

    p = Plots.plot(
        df_yearly.Year, df_yearly.PM,
        label = "PM",
        lw = 2,
        marker = :circle,
        color = :orange,
        xlabel = "Année",
        ylabel = "Médiane de PM",
        title = "PM chez $station",
        legend = :topright,
        margin = 10Plots.mm
    )

    push!(plots, p)
end

Plots.plot(plots..., layout=(2, 4), size=(1800, 600))

Les figures mettent en évidence des différences importantes entre les sites de mesure. De façon générale, plusieurs stations montrent une tendance globale à la baisse au fil du temps, ce qui suggère une diminution progressive des concentrations typiques de particules fines dans l’air à Montréal. Toutefois, cette diminution n’est pas parfaitement régulière, puisque certaines années présentent des remontées ponctuelles.

Les stations les plus anciennes, comme Saint-Jean-Baptiste, l’Aéroport de Montréal et Saint-Joseph, montrent une baisse assez marquée entre le début et la fin de la période observée. Par exemple, à Saint-Jean-Baptiste, la médiane annuelle de $PM$ est élevée au début de la série, puis diminue graduellement malgré quelques fluctuations intermédiaires. L’Aéroport de Montréal présente un comportement similaire, avec des valeurs relativement fortes au début, suivies d’une tendance décroissante plus stable vers les années récentes.

La station Caserne 17 présente également une légère tendance à la baisse, mais avec une variabilité interannuelle plus visible. On observe notamment une hausse marquée autour de $2023$, suivie d’un recul en $2024$. Ce comportement suggère que, même si la tendance générale peut être décroissante, certains épisodes locaux ou certaines conditions particulières peuvent influencer fortement les médianes annuelles.

Pour les stations plus récentes, comme York/Roberval, Saint-Dominique et Anjou, la période d’observation est plus courte, ce qui limite les conclusions que l’on peut tirer. Néanmoins, on remarque que ces stations présentent elles aussi des fluctuations non négligeables d’une année à l’autre. Par exemple, York/Roberval et Anjou montrent une hausse notable en $2023$, suivie d’une baisse en $2024$, ce qui pourrait refléter soit une variabilité annuelle réelle, soit le fait que la série temporelle est encore trop courte pour dégager une tendance robuste.

L’utilisation de la médiane annuelle plutôt que des données journalières permet de mieux faire ressortir les tendances de fond propres à chaque station. En agrégeant les observations à l’échelle annuelle, on réduit l’effet du bruit journalier, des épisodes extrêmes isolés et des fluctuations de très court terme. On obtient ainsi une vision plus claire de l’évolution générale de $PM$ dans le temps, au prix d’une perte d’information sur la variabilité quotidienne.

Cette figure suggère donc que la dynamique de $PM$ dépend non seulement du temps, mais aussi de la localisation de la station. Autrement dit, la station elle-même semble jouer un rôle important dans le niveau typique de particules fines observé. Cela justifie pleinement l’inclusion de l’information spatiale, par exemple l’identifiant ou le nom de la station, dans les futurs modèles prédictifs du projet.

#### 2.3 Moyenne PM par mois
Cette section vise à identifier si certains mois présentent des niveaux de pollution systématiquement plus ou moins élevés, ce qui permettrait d'affiner la précision de nos modèles prédictifs.

In [ ]:
train.mois = month.(train.Date)

moyenne_mensuelle = combine(groupby(train, :mois), :PM => (x -> mean(skipmissing(x))) => :PM_moyen)

bar(moyenne_mensuelle.mois, moyenne_mensuelle.PM_moyen, 
    title="Pollution moyenne par mois (2007-2024)", 
    xlabel="Mois", 
    ylabel="Moyenne IQA (PM)",
    legend=false,
    xticks=1:12,
    color=:skyblue)

On peut observer que les mois ont tous des niveaux de $PM$ relativement similaires, avec des moyennes qui varient entre 15 et 25. Cependant, on remarque que les mois d'été ont tendance à présenter des niveaux de $PM$ légèrement plus bas que les mois d'hiver. Cela peut s'expliquer par le fait que les conditions météorologiques en été, comme une meilleure dispersion de l'air due à des vents plus forts et une plus grande instabilité atmosphérique, peuvent contribuer à réduire la concentration de particules fines. En revanche, en hiver, les conditions peuvent favoriser l'accumulation de $PM$, notamment en raison de l'inversion thermique qui empêche la dispersion des polluants.

#### 2.4 Tendance des polluants
Afin de voir les relations entre la quantité de particules fines et les autres polluants atmosphériques, nous avons créé des graphiques qui représentent l'IQA de $PM$ en fonction de la concentration de $O_3$, $NO_2$ et $SO_2$, selon l'année. Ceci nous permettra de voir si les relations entre ces variables changent au cours du temps.

En premier nous avons fait des graphiques pour voir les relations entre $PM$ et les polluants sur chaque année, afin de voir globalement si les relations entre ces variables changent au cours du temps.

In [ ]:
pollutants = [:O3, :NO2, :SO2]
plots = []

for pollutant in pollutants
    if pollutant in Symbol.(names(train))
        df = dropmissing(train[:, [:Date, :PM, pollutant]])

        df.Year = year.(df.Date)

        df_yearly = combine(groupby(df, :Year),
            :PM => mean => :PM,
            pollutant => mean => pollutant
        )

        fit = safe_r2_simple_regression(df_yearly[!, pollutant], df_yearly.PM)

        if !ismissing(fit)
            β0, β1, r2 = fit

            p = Plots.plot(
                df_yearly.Year, df_yearly.PM,
                label = "PM",
                lw = 2,
                marker = :circle,
                color = :orange,
                xlabel = "Year",
                ylabel = "Mean value",
                title = "PM vs $pollutant (R² = $(round(r2, digits=3)))",
                legend = :topright
            )

            Plots.plot!(
                p,
                df_yearly.Year, df_yearly[!, pollutant],
                label = string(pollutant),
                lw = 2,
                marker = :circle,
                color = :blue
            )

            push!(plots, p)
        end
    end
end

Plots.plot(plots..., layout=(1, length(plots)), size=(1400, 400))

Les graphiques annuels comparant $PM$ aux polluants montrent des comportements assez différents selon la variable considérée. Le coefficient de détermination $R^2$ est faible pour l’ozone, avec $R^2 = 0.032$, ce qui indique qu’il existe peu de relation linéaire entre les moyennes annuelles de PM et de $O_3$. Les deux séries semblent évoluer de manière relativement indépendante, sans tendance commune marquée sur l’ensemble de la période.

La relation entre $PM$ et $NO_2$ est nettement plus forte, avec $R^2 = 0.435$. Les deux courbes présentent une tendance générale semblable, en particulier une diminution progressive au fil du temps, ce qui suggère qu’une partie non négligeable de la variabilité annuelle de PM est associée à celle de $NO_2$. Malgré cela, la relation demeure imparfaite, puisque certaines années montrent des écarts visibles entre les deux séries.

C’est toutefois pour $SO_2$ que le coefficient de détermination est le plus élevé, avec $R^2 = 0.517$. À l’échelle annuelle, les courbes de $PM$ et de $SO_2$ présentent une évolution globale similaire, avec une baisse importante au début de la période suivie d’une stabilisation relative. Cela produit une relation linéaire apparente plus forte que pour les autres polluants lorsque l’on travaille avec les moyennes annuelles.

Il faut cependant interpréter ce résultat avec prudence. Lors d’une analyse précédente effectuée sur les données non agrégées, le $R^2$ associé à $SO_2$ était beaucoup plus faible. Cette différence s’explique par le fait que l’agrégation par année réduit fortement la variabilité journalière ainsi que le bruit à court terme. En moyennant les observations sur une année complète, on élimine une grande partie des fluctuations quotidiennes liées aux conditions météorologiques, aux épisodes ponctuels de pollution et aux variations locales, ce qui fait ressortir davantage les tendances de long terme.

Ainsi, l’analyse annuelle met surtout en évidence des tendances globales communes entre certains polluants et $PM$, tandis que l’analyse sur les données journalières évalue plutôt la capacité d’un polluant à expliquer les fluctuations fines de $PM$ au jour le jour. Dans ce contexte, le contraste observé pour $SO_2$ suggère que son lien avec PM est surtout visible à long terme, mais beaucoup moins clair à l’échelle quotidienne.

#### 2.5 Tendance des polluants par station
On aimerait confirmer que $SO_2$ a bien un relation forte avec $PM$ en comparant leurs IQA à chaque station, pour voir si les fluctuation de $PM$ d'une station à l'autre seraient aussi visibles pour le $SO_2$.

In [ ]:
function pollutantPerStation(pollutant)
    plots = []

    for station in stations
        df = train[coalesce.(train.nom .== station, false), [:Date, :PM, pollutant]]
        df = DataFrames.dropmissing(df)

        df.Year = year.(df.Date)

        df_yearly = combine(groupby(df, :Year),
            :PM => mean => :PM,
            pollutant => mean => pollutant
        )

        fit = safe_r2_simple_regression(df_yearly[!, pollutant], df_yearly.PM)

        if !ismissing(fit)
            β0, β1, r2 = fit
        else
            r2 = 1
        end

        p = Plots.plot(
            df_yearly.Year, df_yearly.PM,
            label = "PM",
            lw = 2,
            marker = :circle,
            color = :orange,
            xlabel = "Année",
            ylabel = "Valeur moyenne",
            title = "PM vs $pollutant chez $station (R² = $(round(r2, digits=3)))",
            legend = :topright,
            margin = 10Plots.mm
        )

        Plots.plot!(
            p,
            df_yearly.Year, df_yearly[!, pollutant],
            label = string(pollutant),
            lw = 2,
            marker = :circle,
            color = :blue
        )

        push!(plots, p)
    end

    Plots.plot(plots..., layout=(2, 4), size=(2200, 600))
end

In [ ]:
pollutantPerStation(:SO2)

Nous possédons des mesures de $SO_2$ uniquement pour les stations Saint-Jean-Baptiste, Saint-Joseph, Saint-Dominique et Anjou. Ce manque de données est problématique pour la robustesse des modèles. 

Les valeurs de $R^2$ calculées semblent cohérentes pour les stations principales. Cependant, pour Saint-Dominique et Anjou, nous observons une instabilité statistique : leurs $R^2$ atteignent une valeur de 1. Ce résultat, théoriquement impossible pour des données environnementales, s'explique par la très faible taille des échantillons de ces stations récentes, rendant l'ajustement linéaire non significatif. Ces limites nous amènent à ne pas retenir le $SO_2$ comme prédicteur dans notre modèle final.

Ces limites concernant le $SO_2$ nous incitent à explorer les autres polluants, comme le $NO_2$, pour vérifier si une situation similaire se reproduit:

In [ ]:
pollutantPerStation(:NO2)

Chaque station comporte des données, ce qui est une bonne nouvelle. Nous pouvons voir que très similairement au $SO_2$, la variation d'une station à l'autre ne semble pas être très significative.

La valeur de $R^2$ semble cohérente entre la valeur présenté en 2.4 et celles des stations plus anciennes (dont nous avons des données datant de 2007). Le même phénomène observée pour la valeur de $R^2$ du $SO_2$ est présent ici, pour les stations Saint-Domique et Anjou, puis York/Roberval possède la même erreur, ce qui est logique, puisque cette station fait aussi partie ce celles dont nous avons moins de valeurs. Nous allons donc encore considéré les $R^2$ de ces trois stations comme des erreurs.

Finalement, voyons la tendance de $O_3$ par station:

In [ ]:
pollutantPerStation(:O3)

Nous pouvons observer que l'IQA du $O_3$ semble plutôt augmenter pour chaque station, ce qui lui donne une tendance inverse au $PM$. Nous pouvons aussi voir que son IQA semble se retrouver dans le même intervale que $PM$, quelque chose qui le différencie des autres polluants, qui possédaient des IQA très faibles comparativement à PM. À certains moments, des hausses de $O_3$ semblent conjoindre avec des hausses de PM, mais globalement, la variation semble très distincte, ce qui augmente notre observation pour ce poluant quant à sa relation avec $PM$ en 2.4.

Comparativement aux deux autres polluants, les valeurs de $R^2$ des stations ne semblent pas cohérent avec celle présenté en 2.4, du moins pour certaines stations. La valeur présenté en 2.4 est de 0.032, ce qui se rapproche des stations Saint-Joseph et Caserne 17, mais est grandement inférieur à celle pour les stations Saint-Jean-Baptiste et Aéroport de Montréal, indiquant que $O_3$ à un plus grand impact sur $PM$ pour ces stations. Enfin, le même résultat de $R^2$ est observé ici pour les trois stations ayant le moins de données. Nous allons encore une fois identifié ces valeurs de $R^2$ comme des erreurs.

Pour les valeurs de $R^2$ identifiés auparavant, chacune avait été calculées avec la moyenne de chaque année. Nous aimerions maintenant nous intéresser aux valeurs de $R^2$ calculés à l'aide de toutes les données disponible, pour évalué comment ces valeurs vont se comparer aux moyennes :

In [ ]:
pollutants = [:O3, :NO2, :SO2]
plots = []


for pollutant in pollutants
    if pollutant in Symbol.(names(train))
        fit = safe_r2_simple_regression(train[!, pollutant], train.PM)


        if !ismissing(fit)
            β0, β1, r2 = fit


            valid_idx = .!ismissing.(train.PM) .& .!ismissing.(train[!, pollutant])
            x_data = Float64.(train[valid_idx, pollutant])
            y_data = Float64.(train[valid_idx, :PM])


            p = scatter(x_data, y_data,
                xlabel = string(pollutant),
                ylabel = "PM",
                title = "PM vs $pollutant (R²=$(round(r2, digits=3)))",
                legend = false,
                markersize = 2,
                alpha = 0.3,
                markercolor = :steelblue)


            x_line = range(minimum(x_data), maximum(x_data), length=200)
            y_line = β0 .+ β1 .* x_line


            Plots.plot(p, x_line, y_line,
                color = :red,
                linewidth = 2)


            push!(plots, p)
        end
    end
end


Plots.plot(plots..., layout=(1, length(plots)), size=(1400, 400))

$NO_2$ ($R^2$ ≈ 0,172) : a régression linéaire simple avec $NO_2$ explique environ 17,2% de la variabilité de $PM$, ce qui en fait le meilleur des trois prédicteurs considérés ici dans un cadre linéaire simple. Le nuage de points suggère une tendance croissante réelle, mais les valeurs de $NO_2$ semblent aussi regroupées en bandes ou en niveaux discrets, ce qui peut refléter un arrondi ou une quantification des mesures et rendre l’interprétation visuelle moins nette; de plus, la dispersion demeure importante, donc le pouvoir explicatif de $NO_2$ seul reste modéré plutôt qu’élevé.


$O_3$ ($R^2$ ≈ 0,001) : la régression linéaire simple avec $O_3$ n’explique pratiquement aucune variabilité de $PM$. Le nuage de points est très dispersé et la pente estimée est presque nulle, ce qui indique que $O_3$ seul a très peu d’utilité pour un modèle linéaire simple de $PM$.

$SO_2$ ($R^2$≈ 0,004) : la régression linéaire simple avec $SO_2$ n’explique qu’une proportion négligeable de la variabilité de $PM$. Même si la droite ajustée a une légère pente positive, le très faible montre que l’ajustement linéaire est très pauvre et qu’il n’y a pas de relation linéaire simple utile entre $SO_2$ et $PM$ dans ces données.

#### 2.2 Quantité de particules fines en fonction de la vitesse moyenne du vent, par station


In [ ]:
plots = []

for station in stations
    df = train[coalesce.(train.nom .== station, false), [:Date, :PM, :vitesse_vent_moy]]
    df = DataFrames.dropmissing(df)

    df.Year = year.(df.Date)

    df_yearly = combine(groupby(df, :Year),
        :vitesse_vent_moy => mean => :vitesse_vent_moy
    )

    p = Plots.plot(
        df_yearly.Year, df_yearly.vitesse_vent_moy,
        label = "Vitesse du vent",
        lw = 2,
        marker = :circle,
        color = :blue,
        xlabel = "Année",
        ylabel = "Valeur moyenne",
        title = "Vitesse du vent chez $station",
        legend = :topright,
        margin = 10Plots.mm
    )

    push!(plots, p)
end

Plots.plot(plots..., layout=(2,4), size=(2000,600))

On observe globalement que la vitesse du vent varie au fil du temps. Elle semble varier de la même manière d'une station à l'autre.

Si on compare ces résultats avec la tendance de $PM$ en 2.2, nous pouvons visuellement déterminer que la relation entre les deux variables semble faible. Les hausses et les baisses de $PM$ ne suivent pas de façon claire celles de la vitesse moyenne du vent, ce qui suggère qu’une régression simple basée uniquement sur cette variable météorologique possède un pouvoir explicatif limité.

Les graphiques suggèrent que la vitesse moyenne du vent, prise seule, n’est probablement pas suffisante pour expliquer adéquatement les variations de $PM$. Il sera donc pertinent d’examiner d’autres variables météorologiques et de considérer des modèles multivariés pour améliorer la prédiction.

### 2.3 Corrélations entre PM et les variables numériques



In [ ]:
results = DataFrame(
    Variable = String[],
    Intercept = Float64[],
    Slope = Float64[],
    R2 = Float64[]
)

exclude_cols = ["Date", "nom", "stationId", "PM"]

for col in names(train)
    if !(col in exclude_cols) && eltype(train[!, col]) <: Union{Missing, Number}
        fit = safe_r2_simple_regression(train[!, col], train.PM)

        if !ismissing(fit)
            β0, β1, r2 = fit
            push!(results, (col, β0, β1, r2))
        end
    end
end

sort!(results, :R2, rev=true)

println("\n=== TOP 15 VARIABLES SELON LE R² DE LA RÉGRESSION SIMPLE AVEC PM ===")
println(first(results, 15))

top_n = min(20, nrow(results))
top_results = first(results, top_n)

bar(top_results.Variable, top_results.R2,
    xlabel="Variables explicatives",
    ylabel="R²",
    title="Top $top_n variables selon le R² de la régression simple avec PM",
    legend=false,
    fillcolor=:steelblue,
    size=(900, 700),
    xrotation=45)

Ici, on peut observer que les variables comme la concentration de NO2, la visibilité et le vent ont une forte corrélation avec la quantité de particules fines. Tandis que, des variables comme la température et la pression ont une corrélation plus faible avec la quantité de particules fines car elles pourraient ne pas être directement liées à la quantité de particules fines dans l'air. Par exemple, la température peut influencer la formation de particules fines, mais elle n'est pas un facteur direct de leur concentration.

### 2.5 Relations PM vs Variables météorologiques
Dans cette section, nous explorons les liens entre l’indice de particules fines (PM) et plusieurs variables météorologiques, notamment la température, l’humidité relative, la vitesse du vent, la pression, les précipitations totales et la visibilité. Cette analyse exploratoire permet de visualiser la forme des relations, de vérifier si une tendance linéaire semble plausible et d’identifier quelles variables pourraient être utiles pour la prédiction de PM dans la suite du projet

In [ ]:
meteo_vars = [:temp_moy, :hum_rel_moy, :vitesse_vent_moy, :pression_moy,
              :precip_tot, :visibilite_moy, :visibilite_max, :visibilite_min, 
              :vitesse_vent_max, :vitesse_vent_min, :pression_max, :pression_min]

plots_meteo = []

for var in meteo_vars
    if var in Symbol.(names(train))
        fit = safe_r2_simple_regression(train[!, var], train.PM)

        if !ismissing(fit)
            β0, β1, r2 = fit

            df_plot = dropmissing(train[:, [:Date, :PM, var]])

            x_range = range(minimum(df_plot[!, var]), maximum(df_plot[!, var]), length=100)
            y_range = β0 .+ β1 .* x_range

            p = Plots.scatter(
                df_plot[!, var], df_plot.PM,
                label        = nothing,
                ms           = 2,
                alpha        = 0.2,
                color        = :orange,
                xlabel       = string(var),
                ylabel       = "PM",
                title        = "PM vs $(var)\n(R² = $(round(r2, digits=3)))",
                legend       = :topright,
                titlefontsize = 9
            )

            Plots.plot!(p,
                x_range, y_range,
                label  = "Régression",
                lw     = 2,
                color  = :blue
            )

            push!(plots_meteo, p)
        end
    end
end

n     = length(plots_meteo)
ncols = 3
nrows = ceil(Int, n / ncols)

Plots.plot(plots_meteo..., 
    layout = (nrows, ncols), 
    size   = (1400, nrows * 350)
)

Les graphiques annuels comparant PM aux variables météorologiques montrent que les relations linéaires simples demeurent globalement faibles. Les coefficients de détermination $R^2$ restent petits, ce qui indique que chaque variable prise individuellement explique seulement une faible part de la variabilité de PM. En régression linéaire, le coefficient $R^2$ mesure la proportion de la variabilité de la variable réponse expliquée par le modèle, et plus il est proche de 1, meilleur est l’ajustement.

Parmi les variables étudiées, la visibilité moyenne est celle qui présente la relation linéaire simple la plus marquée avec PM, avec $R^2 \approx 0{,}067$, suivie de la vitesse moyenne du vent avec $R^2 \approx 0{,}060$. Même à l’échelle annuelle, ces valeurs demeurent faibles, ce qui montre que leur pouvoir explicatif reste limité dans un modèle univarié. Visuellement, on remarque toutefois que les années où PM est plus élevé correspondent souvent à des niveaux de visibilité plus faibles et à des vitesses moyennes du vent plus basses, ce qui est cohérent avec une dispersion atmosphérique moins efficace des particules fines.

L’humidité relative moyenne présente une relation plus faible avec PM, avec $R^2 \approx 0{,}025$. Sur le graphique annuel, les deux séries montrent parfois des variations de même ordre, mais la concordance demeure irrégulière d’une année à l’autre. Cela suggère que l’humidité seule n’explique qu’une faible fraction de la variabilité annuelle de PM.

La température moyenne, la pression moyenne et les précipitations totales montrent quant à elles des relations linéaires très faibles avec PM, avec des $R^2$ d’environ $0{,}004, 0{,}002$  et $0{,}001$ respectivement. Les courbes annuelles correspondantes ne suivent pas de tendance commune nette avec celle de PM, ce qui confirme que ces variables, considérées individuellement, apportent peu d’information dans une régression linéaire simple.

On observe également que certaines années se distinguent davantage que d’autres, avec des écarts marqués pour PM ou pour certaines variables météorologiques. Ce type de variation peut compliquer l’interprétation, surtout si les hypothèses de linéarité ou d’homoscédasticité ne sont pas parfaitement respectées.

Dans l’ensemble, cette analyse exploratoire à l’échelle annuelle suggère que les variables météorologiques prises séparément expliquent seulement une petite partie de la variabilité de PM. Elles pourraient néanmoins devenir plus utiles dans un modèle de régression multiple, où leur effet combiné serait étudié simultanément avec d’autres variables explicatives.

### 2.7 Analyse par jours de la semaine des particules fines

Cette section vise à identifier si certains jours de la semaine présentent des niveaux de pollution systématiquement plus ou moins élevés, ce qui permettrait d'affiner la précision de nos modèles prédictifs.


In [ ]:
train.jour_num = dayofweek.(train.Date)
train.is_weekend = [d >= 6 ? 1 : 0 for d in train.jour_num]
train.is_weekend_and_friday = [d >= 5 ? 1 : 0 for d in train.jour_num]

analyse_par_jour = combine(groupby(train, :jour_num), 
    :PM => (x -> mean(skipmissing(x))) => :PM_moyen,
    :NO2 => (x -> mean(skipmissing(x))) => :NO2_moyen
)

noms_jours = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
analyse_par_jour.Nom = noms_jours[analyse_par_jour.jour_num]

sort!(analyse_par_jour, :jour_num)
println("--- TABLEAU DES PM PAR JOUR ---")
display(analyse_par_jour[:, [:jour_num, :Nom, :PM_moyen, :NO2_moyen ]])


jours = analyse_par_jour.Nom
pm_vals = analyse_par_jour.PM_moyen
no2_vals = analyse_par_jour.NO2_moyen



fit_jour_num = safe_r2_simple_regression(train.jour_num, train.PM)
fit_weekend = safe_r2_simple_regression(train.is_weekend, train.PM)
fit_weekend_and_friday = safe_r2_simple_regression(train.is_weekend_and_friday, train.PM)
println("\n=== R² DE LA RÉGRESSION SIMPLE AVEC JOUR_NUM ===")
println("R² : $(fit_jour_num)")

println("\n=== R² DE LA RÉGRESSION SIMPLE AVEC IS_WEEKEND ===")
println("R² : $(fit_weekend)")

println("\n=== R² DE LA RÉGRESSION SIMPLE AVEC IS_WEEKEND_ANDFRID ===")
println("R² : $(fit_weekend_and_friday)")

On peut observer que les jours de la semaine présentent des niveaux de PM relativement stables, avec des moyennes qui oscillent étroitement autour de 19 à 20. Cependant, on remarque que le samedi affiche le niveau le plus bas (18.94), tandis que le milieu de semaine (mardi et mercredi) marque les pics de concentration.
Cela peut s'expliquer par le rythme des activités humaines : la baisse du trafic routier lourd et la réduction de l'activité industrielle durant le week-end contribuent à une diminution des NO2 et légèrement, le PM. L'augmentation de PM le dimanche peut être expliqué par le chauffage résidentiel ou les activités de loisirs.

Vu les différences assez significatives entre les jours, les jours de la semaine(jours_num) peuvent être considérés comme une variable explicative dans le modèle. Toutefois, le R2 calculé semble montrer le contraire. La variable du is_weekend_and_friday est légèrement meilleur dans le domaine de R2 et pourrait être considéré pour augmenter légèrement la précision du modèle.

### 2.8 Analyse par veille de PM


Cette section vise à identifier si le PM de la veille présente une corrélation avec le PM d'aujourd'hui, ce qui permettrait d'affiner la précision de nos modèles prédictifs.

In [ ]:
sort!(train, :Date)
train.PM_veille = [missing; train.PM[1:end-1]]

clean_df = dropmissing(train, [:PM, :PM_veille])
println("\n=== R² DE AVEC PM_veille ===")
fit = safe_r2_simple_regression(clean_df.PM_veille, clean_df.PM)
println("Fitted parameters : $(fit)")

x = clean_df.PM_veille
y = clean_df.PM
intercept = fit[1]
slope = fit[2]

scatter(x, y, 
    alpha=0.5, 
    markerstrokewidth=0, 
    label="Données journalières",
    title="Corrélation entre PM(t-1) et PM(t)",
    xlabel="PM de la veille (µg/m³)",
    ylabel="PM d'aujourd'hui (µg/m³)",
    legend=:topleft
)

display(plot!(x, intercept .+ slope .* x, 
    color=:red, 
    linewidth=3, 
    label="Ligne de régression (R² = 0.437)"))


scatter(x, y, 
    alpha=0.4,            
    markerstrokewidth=0, 
    label="Données journalières",
    title="Corrélation PM(t-1) vs PM(t) (Zoom < 200)",
    xlabel="PM de la veille (µg/m³)",
    ylabel="PM d'aujourd'hui (µg/m³)",
    legend=:topleft,
    xlims=(0, 200),         
    ylims=(0, 200)          
)

x_range = 0:200
plot!(x_range, intercept .+ slope .* x_range, 
    color=:red, 
    linewidth=3, 
    label="Tendance (R² = 0.437)")

Le graphique met en évidence une corrélation positive marquée entre le PM d'aujourd'hui et le PM de la veille. Celle-ci est confirmé par son score R2 de 0.437, particulièrement significatif pour la variable. 
Il démontre qu'à elle seule, la valeur de la veille explique près de 44 % de la variance des niveaux de pollution actuels. Dans un système aussi complexe et chaotique, obtenir une telle précision avec une variable unique est un indicateur de fiabilité majeur.

Ceci peut être expliqué par les particules fines possédant une certaine inertie dans la région. Elles restent en suspension, créant une continuité temporelle : si l'air est saturé un jour donné, une partie de cette pollution stagne et "déborde" sur le lendemain.

De plus, certaines sources de pollution ne s'allument et ne s'éteignent pas de manière aléatoire d'un jour à l'autre, créant un effet "plateau" de pollution, où la pollution s'accumule et se maintient à des niveaux similaires tant que les conditions météorologiques ou les rythmes de production ne changent pas radicalement.

### 2.8 Analyse par $PM_{t-2}$ de PM

Cette section vise à identifier si le $PM_{t-2}$ une corrélation avec le PM d'aujourd'hui. En effet, si $PM_{t-1}$ donne une corrélation assez significative, $PM_{t-2}$ pourrait possiblement aussi avoir une influence pertinante.

In [ ]:
sort!(train, :Date)
train.PM_t2 = [missing; missing; train.PM[1:end-2]]

clean_df = dropmissing(train, [:PM, :PM_veille, :PM_t2])

println("\n=== R² DE AVEC PM_t2 ===")
fit = safe_r2_simple_regression(clean_df.PM_t2, clean_df.PM)
println("Fitted parameters : $(fit)")

scatter(clean_df.PM_t2, clean_df.PM, 
    alpha=0.4, 
    xlims=(0,200), ylims=(0,200),
    title="Corrélation PM(t-2) vs PM(t)",
    xlabel="PM d'avant-hier (t-2)",
    ylabel="PM d'aujourd'hui (t)",
    label="Données t-2",
    color=:green)

x_range = 0:200
plot!(x_range, fit[1] .+ fit[2] .* x_range, 
    color=:darkgreen, linewidth=3, label="Tendance t-2")

L'analyse de la variable $PM_{t-2}$ (l'avant-veille) marque une étape décisive dans la compréhension de la dynamique des particules fines pour notre modèle. Bien que moins intuitive que la veille immédiate, cette variable apporte une profondeur temporelle essentielle. Le score R2 individuel de 0,3533 pour $PM_{t-2}$ démontre que l'air de la région conserve une "mémoire" significative de sa pollution même après 48 heures, rendant les épisodes de pollution plus stables et prévisibles.

Bien que l'utilisation de $PM_{t-2}$ comme variable explicative soit statistiquement valide et pertinente, son inclusion dans un modèle de régression linéaire classique exige une certaine prudence. La forte corrélation entre $PM_{t-1}$ et $PM_{t-2}$ introduit un risque de multicolinéarité.

### Conclusion de la section

En conclusion, cette phase d'exploration a permis de valider la structure de nos données et d'identifier les variables les plus pertinentes pour la modélisation. Bien que le $SO_2$ ait été initialement considéré, les instabilités statistiques observées (notamment des valeurs de $R^2$ non significatives dues à de faibles échantillons) et le manque de données pour plusieurs stations nous ont conduits à l'écarter. Nous concentrerons donc la suite de notre étude sur le $NO_2$ et les variables météorologiques, qui présentent des corrélations plus stables et un pouvoir explicatif plus fiable pour prédire les concentrations de particules fines ($PM$). 

---
## 3. Préparation des données et ingénierie des variables

L'analyse exploratoire a mis en évidence plusieurs points que nous allons exploiter directement dans la construction des variables :

- La station et le temps de l'année (saison) semble avoir une grande relation.
- PM de la veille fortement corrélé avec PM du jour courant : c'est la variable la plus informative que nous trouvons intéressant à aller explorer.
- SO₂ complètement absent pour 3 stations sur 7 : nous décidons donc de l'enlever de nos variables explicatives, par peur que le manque de donnée apporte des complications.
- O₃ disponible pour toutes les stations : nous l'incluons dans nos modèles ;
- Relations non-linéaires et saisonnières : nous allons utiliser un encodage cyclique $\sin$/$\cos$ du jour de l'année, ce qui nous permet d'avoir un plus grand échantillon de donnée par valeur.

### 3.1 Variables autorégressives : PM à la veille, par station

Dans la version précédente, nous avions calculé `PM_veille` en triant l'ensemble des observations par date, sans séparer par station. Le $R^2$ de $0{,}437$ que nous avions obtenu était donc surestimé : on comparait le PM d'une station à celui d'une autre station le jour d'avant. En calculant correctement le décalage **par station**, le $R^2$ univarié redescend à environ $0{,}21$, ce qui reste plus élevé que pour toute autre variable individuelle. Il devient donc très intéressant de l'inclure dans nos modèles.


In [ ]:
sort!(train, [:stationId, :Date])

function add_pm_lags!(df::DataFrame)
    df.PM_lag1 = Vector{Union{Missing, Float64}}(missing, nrow(df))
    for sdf in groupby(df, :stationId)
        rows = parentindices(sdf)[1]
        vals = df.PM[rows]
        lagged = Vector{Union{Missing, Float64}}(undef, length(rows))
        lagged[1] = missing
        for i in 2:length(rows)
            lagged[i] = vals[i-1]
        end
        df.PM_lag1[rows] = lagged
    end

    df.logPM_lag1 = Vector{Union{Missing, Float64}}(missing, nrow(df))
    for i in 1:nrow(df)
        v = df.PM_lag1[i]
        df.logPM_lag1[i] = ismissing(v) || v <= 0 ? missing : log(v)
    end
    return df
end

add_pm_lags!(train)

clean = dropmissing(train, [:PM, :logPM_lag1])
r2_lag = cor(clean.logPM_lag1, log.(clean.PM))^2
println("R² de logPM_lag1 sur log(PM) (par station) : ",
        round(r2_lag, digits=3))
println("Observations avec lag disponible : ", nrow(clean), " / ", nrow(train))

### 3.2 Encodage cyclique saisonnier

L'analyse de la section 2.6 a montré une saisonnalité douce : PM légèrement plus bas l'été, plus élevé en hiver. Au lieu d'utiliser 12 indicatrices pour les différents mois, nous plutôt utiliser ce système qui divise nos données en 2 modèles :

$$s_1(t) = \sin\!\left(\frac{2\pi \cdot \text{doy}(t)}{365{,}25}\right), \quad c_1(t) = \cos\!\left(\frac{2\pi \cdot \text{doy}(t)}{365{,}25}\right)$$

avec 2 deuxième groupe de 2 modèles qui capture la variance :

$$s_2(t) = \sin\!\left(\frac{4\pi \cdot \text{doy}(t)}{365{,}25}\right), \quad c_2(t) = \cos\!\left(\frac{4\pi \cdot \text{doy}(t)}{365{,}25}\right).$$

Où doy := jour de l'année (Day Of Year)

Quatre variables continues remplacent ainsi 11 indicatrices mensuelles, tout en respectant la continuité du temps.

In [ ]:
function add_time_features!(df::DataFrame)
    df.mois        = month.(df.Date)
    df.mois_str    = string.(df.mois)
    df.station_str = string.(df.stationId)
    df.annee       = year.(df.Date)
    df.doy         = dayofyear.(df.Date)
    df.jour_num    = dayofweek.(df.Date)
    df.is_weekend  = Int.(df.jour_num .>= 6)

    df.sin_doy  = sin.(2π .* df.doy ./ 365.25)
    df.cos_doy  = cos.(2π .* df.doy ./ 365.25)
    df.sin_doy2 = sin.(4π .* df.doy ./ 365.25)
    df.cos_doy2 = cos.(4π .* df.doy ./ 365.25)
    return df
end

add_time_features!(train)
println("Colonnes ajoutées : mois, mois_str, ",
        "station_str, annee, doy, jour_num, is_weekend, sin_doy, cos_doy, ",
        "sin_doy2, cos_doy2")

### 3.3 Variables météorologiques dérivées

Nous enrichissons le jeu de données avec des **amplitudes journalières** (max − min) qui mesurent la variabilité intra-journée, utile pour détecter des journées à forte instabilité atmosphérique :

In [ ]:
function add_meteo_derived!(df::DataFrame)
    df.temp_amp     = df.temp_max     .- df.temp_min
    df.hum_amp      = df.hum_rel_max  .- df.hum_rel_min
    df.pression_amp = df.pression_max .- df.pression_min
    df.vent_amp     = df.vitesse_vent_max .- df.vitesse_vent_min
    df.visib_amp    = df.visibilite_max .- df.visibilite_min
    return df
end

add_meteo_derived!(train)
println("Amplitudes journalières ajoutées.")

### 3.4 Imputation avec indicateurs de manquance

L'analyse exploratoire a révélé que les stations Aéroport de Montréal, York/Roberval et Caserne 17 n'ont aucune mesure de SO₂. Nous avons décider de ne pas prendre en compte le $SO_2$, en gros par son manque de relation avec PM, ainsi que l'inquiétude que le manque de données vont apporté à des problématiques.

Pour les autres variables manquantes (NO₂, O₃, météo), nous appliquons :

1. Un indicateur binaire de manquance (_miss) pour chaque polluant, cela permet au modèle d'apprendre un effet propre aux observations imputées.
2. Une imputation par la médiane conditionnelle (stationId × mois), qui respecte les effets spatiaux et saisonniers observés en 2.2 et 2.6.

In [ ]:
for var in [:NO2, :O3, :SO2]
    train[!, Symbol(string(var) * "_miss")] = Int.(ismissing.(train[!, var]))
end

function imputer_par_station_mois!(df, ref_df, var; group_col=:stationId, time_col=:Date)
    if eltype(df[!, var]) <: Union{Missing, Integer}
        df[!, var] = convert(Vector{Union{Missing, Float64}}, df[!, var])
    end

    vals_global = collect(skipmissing(ref_df[!, var]))
    isempty(vals_global) && return
    median_global = median(vals_global)

    ref_clean = dropmissing(ref_df[:, [:station_str, :mois, var]])
    lookup = Dict{Tuple{String, Int}, Float64}()
    if nrow(ref_clean) > 0
        grouped = combine(groupby(ref_clean, [:station_str, :mois]), var => median => :med_grp)
        lookup = Dict((row.station_str, row.mois) => Float64(row.med_grp) for row in eachrow(grouped))
    end

    sort!(df, [group_col, time_col])
    for sdf in groupby(df, group_col)
        rows_idx = parentindices(sdf)[1]
        prev = missing
        for i in rows_idx
            if ismissing(df[i, var])
                key = (df.station_str[i], df.mois[i])
                df[i, var] = ismissing(prev) ? get(lookup, key, median_global) : prev
            end
            prev = df[i, var]
        end
    end
end

vars_to_impute = [:NO2, :O3, :temp_moy, :hum_rel_moy, :pression_moy,
                  :visibilite_moy, :vitesse_vent_moy, :neige_au_sol,
                  :neige, :pluie, :precip_tot]

for v in vars_to_impute
    if v in Symbol.(names(train))
        imputer_par_station_mois!(train, train, v)
    end
end

imputer_par_station_mois!(train, train, :SO2)

mu_NO2  = mean(train.NO2)
mu_temp = mean(train.temp_moy)
mu_vis  = mean(train.visibilite_moy)
mu_hum  = mean(train.hum_rel_moy)
mu_vent = mean(train.vitesse_vent_moy)

train.NO2_c            = train.NO2 .- mu_NO2
train.temp_moy_c       = train.temp_moy .- mu_temp
train.visibilite_moy_c = train.visibilite_moy .- mu_vis
train.hum_rel_moy_c    = train.hum_rel_moy .- mu_hum
train.vent_c           = train.vitesse_vent_moy .- mu_vent

train.NO2_c_sq            = train.NO2_c .^ 2
train.temp_moy_c_sq       = train.temp_moy_c .^ 2
train.visibilite_moy_c_sq = train.visibilite_moy_c .^ 2

first(train, 5)

### 3.5 Interactions physiques

La dispersion atmosphérique des polluants est amplifiée par le vent : un taux de NO₂ élevé lorsque la vitesse moyenne du vent est plus faible correspond à une pollution stagnante, alors que le même taux lorsque cette même vitesse est plus élevée se traduit par une pollution qui se disperse et affecte moins PM. Nous introduisons donc le terme d'interaction $\mathrm{NO2_c} \times \mathrm{vent_c}$.

De la même façon, l'effet de la température sur la formation secondaire de particules dépend de l'humidité, d'où une interaction $\mathrm{temp\_moy_c} \times \mathrm{hum\_rel\_moy_c}$.

In [ ]:
train.NO2_x_vent    = train.NO2_c .* train.vent_c
train.temp_x_hum    = train.temp_moy_c .* train.hum_rel_moy_c
println("Interactions physiques ajoutées : NO2_x_vent, temp_x_hum")

### 3.6 Détection et traitement des valeurs suspectes

Nous détectons les observations dont le résidu standardisé dépasse 3 en valeur absolue. Nous les écartons de l'ajustement final pour limiter leur influence sur les coefficients. Pour déterminer le résidu, nous avons développer un modèle de régression linéaire envers le logarithme de PM et les quelques variables explicatives dont nous avons mentionner dans cette section.

In [ ]:
ref_formula = @formula(log(PM) ~ station_str + sin_doy + cos_doy + sin_doy2 + cos_doy2 +
                                  NO2_c + NO2_c_sq + temp_moy_c + temp_moy_c_sq +
                                  visibilite_moy_c + visibilite_moy_c_sq + vent_c +
                                  hum_rel_moy_c + pression_moy + O3 +
                                  pluie + neige + neige_au_sol)

train_fit = dropmissing(train, :PM)
m_ref = lm(ref_formula, train_fit)
residus_brut = residuals(m_ref)
sigma_hat = sqrt(sum(residus_brut.^2) / dof_residual(m_ref))
residus_std = residus_brut ./ sigma_hat

println("Observations utilisables : ", nrow(train_fit))
println("Résidus standardisés |.| > 3 : ", sum(abs.(residus_std) .> 3),
        " (", round(100*sum(abs.(residus_std) .> 3)/length(residus_std), digits=2), " %)")

mask_clean = abs.(residus_std) .<= 3
train_clean = train_fit[mask_clean, :]
println("Après nettoyage : ", nrow(train_clean), " observations")

---
## 4. Modèles de régression

Nous comparons trois familles de modèles :

1. **Modèle linéaire de référence** (modèle 5 de la version précédente), remis au propre avec les nouvelles variables.
2. **Modèle autorégressif** : même forme, mais avec `logPM_lag1` — variable la plus informative identifiée.
3. **Régression ridge bayésienne** (Ch. 7) avec le meilleur $\lambda$ choisi par **BIC** (Ch. 7, section 7.6 du cours).

### 4.1 Fonctions utilitaires

In [ ]:
rmse(y, yhat) = sqrt(mean((y .- yhat).^2))

# ---- Modèle 5 amélioré (sans lag) : version propre et corrigée -------------
function model_baseline(df::DataFrame)
    return lm(@formula(log(PM) ~ station_str + sin_doy + cos_doy + sin_doy2 + cos_doy2 +
                                  NO2_c + NO2_c_sq + temp_moy_c + temp_moy_c_sq +
                                  visibilite_moy_c + visibilite_moy_c_sq +
                                  vent_c + hum_rel_moy_c + pression_moy + O3 +
                                  pluie + neige + neige_au_sol +
                                  NO2_x_vent + temp_x_hum +
                                  NO2_miss + O3_miss + SO2_miss), df)
end

# ---- Modèle autorégressif : ajoute logPM_lag1 (AMÉLIORATION A) -------------
function model_ar(df::DataFrame)
    return lm(@formula(log(PM) ~ logPM_lag1 +
                                  station_str + sin_doy + cos_doy + sin_doy2 + cos_doy2 +
                                  NO2_c + NO2_c_sq + temp_moy_c + temp_moy_c_sq +
                                  visibilite_moy_c + visibilite_moy_c_sq +
                                  vent_c + hum_rel_moy_c + pression_moy + O3 +
                                  pluie + neige + neige_au_sol +
                                  NO2_x_vent + temp_x_hum +
                                  NO2_miss + O3_miss + SO2_miss), df)
end

### 4.2 Régression ridge avec sélection de $\lambda$ par BIC (Ch. 7)

La section 7.6 du chapitre 7 présente le **critère bayésien d'information (BIC)** pour choisir entre plusieurs modèles de régression. Pour une régression linéaire avec $n$ observations et $p_\gamma$ variables dans le modèle $M_\gamma$ :

$$\mathrm{BIC}(M_\gamma) = n\ln\!\left(\frac{\mathrm{SSE}_\gamma}{n}\right) + p_\gamma \ln(n).$$

Le meilleur modèle est celui qui **minimise le BIC**. (Le cours utilise la convention « maximiser », mais les deux conventions sont équivalentes au signe près — ici nous minimisons.)

Pour la régression ridge, le nombre effectif de paramètres n'est pas $p$ mais le **degré de liberté effectif** :

$$\mathrm{df}(\lambda) = \mathrm{tr}\!\left(X(X^\top X + \lambda I)^{-1} X^\top\right).$$

Cette quantité varie de $p$ (quand $\lambda = 0$, ridge = OLS) à $0$ (quand $\lambda \to \infty$). Nous utilisons le BIC ridge :

$$\mathrm{BIC}_{\text{ridge}}(\lambda) = n\ln\!\left(\frac{\mathrm{SSE}(\lambda)}{n}\right) + \mathrm{df}(\lambda) \ln(n).$$

In [ ]:
# AMÉLIORATION F : Construction de la matrice de design et ridge + BIC

function build_design_matrix(df::DataFrame;
                              use_lag::Bool = true,
                              reference_levels = nothing)
    cont_vars = Symbol[:NO2_c, :NO2_c_sq,
                        :temp_moy_c, :temp_moy_c_sq,
                        :visibilite_moy_c, :visibilite_moy_c_sq,
                        :vent_c, :hum_rel_moy_c, :pression_moy,
                        :O3, :pluie, :neige, :neige_au_sol,
                        :sin_doy, :cos_doy, :sin_doy2, :cos_doy2,
                        :NO2_x_vent, :temp_x_hum,
                        :NO2_miss, :O3_miss, :SO2_miss]

    if use_lag
        pushfirst!(cont_vars, :logPM_lag1)
    end

    cat_vars = [:station_str]
    n = nrow(df)

    cols = [ones(n)]
    names_out = String["Intercept"]

    for v in cont_vars
        x = Float64.(df[!, v])
        push!(cols, x)
        push!(names_out, string(v))
    end

    levels_used = Dict{Symbol, Vector{String}}()
    for v in cat_vars
        if reference_levels !== nothing && haskey(reference_levels, v)
            lvls = reference_levels[v]
        else
            lvls = sort(unique(df[!, v]))
        end
        levels_used[v] = lvls
        for lvl in lvls[2:end]
            push!(cols, Float64.(df[!, v] .== lvl))
            push!(names_out, string(v) * "_" * lvl)
        end
    end

    X = hcat(cols...)
    return X, names_out, levels_used
end

# --- Ridge en forme fermée ---
function ridge_fit(X, y, lambda)
    p = size(X, 2)
    L = Diagonal([0.0; fill(lambda, p-1)])  # pas de pénalité sur l'intercept
    return (X' * X + L) \ (X' * y)
end

# --- BIC pour la régression ridge ---
function ridge_bic(X, y, lambda)
    n, p = size(X)
    β    = ridge_fit(X, y, lambda)
    ŷ    = X * β

    # Degré de liberté effectif de la régression ridge :
    #   df(λ) = tr(X (X'X + λI)⁻¹ X') = tr((X'X + λI)⁻¹ X'X)
    # La deuxième forme évite de construire la matrice n×n.
    L = Diagonal([0.0; fill(lambda, p-1)])
    XtX = X' * X
    df_eff = tr((XtX + L) \ XtX)

    sse = sum((y .- ŷ).^2)
    bic = n * log(sse / n) + df_eff * log(n)
    return bic, df_eff, sse
end

println("Fonctions ridge + BIC définies.")

### 4.3 Choix de $\lambda$ par BIC

Nous explorons une grille géométrique de $\lambda$ entre $10^{-3}$ et $10^3$, et sélectionnons celui qui minimise le BIC.

In [ ]:
# Construction de X et y pour le modèle ridge avec lag
train_ar = dropmissing(train_clean, [:PM, :logPM_lag1])
X_tr, var_names, ref_levels = build_design_matrix(train_ar, use_lag=true)
y_tr = log.(Float64.(train_ar.PM))

# Standardisation (hors intercept) — important pour que λ affecte les variables
# de manière comparable (section 7.5 du cours)
mu_X    = vec(mean(X_tr[:, 2:end], dims=1))
sigma_X = vec(std(X_tr[:, 2:end], dims=1))
sigma_X[sigma_X .== 0] .= 1.0

function standardize(X, mu, sigma)
    Xs = copy(X)
    Xs[:, 2:end] = (X[:, 2:end] .- mu') ./ sigma'
    return Xs
end

X_tr_std = standardize(X_tr, mu_X, sigma_X)

# Recherche du meilleur λ par BIC
lambda_grid = [10.0^k for k in -3:0.25:3]
bic_scores = Float64[]
df_effs    = Float64[]
for λ in lambda_grid
    bic, df_eff, sse = ridge_bic(X_tr_std, y_tr, λ)
    push!(bic_scores, bic)
    push!(df_effs, df_eff)
end

best_idx    = argmin(bic_scores)
best_lambda = lambda_grid[best_idx]

println("Dimensions de la matrice de design : ", size(X_tr_std))
println("Meilleur λ par BIC : ", round(best_lambda, sigdigits=4))
println("BIC correspondant : ", round(bic_scores[best_idx], digits=2))
println("Degré de liberté effectif : ", round(df_effs[best_idx], digits=2),
        " (sur ", size(X_tr_std, 2), " variables)")

# Ajustement final
β_ridge = ridge_fit(X_tr_std, y_tr, best_lambda)
println("\nCoefficients ridge ajustés (première/dernière valeur) :")
for (i, nm) in enumerate(var_names[1:min(6, end)])
    println("  ", nm, " = ", round(β_ridge[i], digits=4))
end

### 4.4 Validation rolling-origin

Comme dans la version précédente, on valide sur les années 2021-2024 en entraînant sur toutes les années antérieures à l'année de validation. C'est l'estimation la plus fidèle de la performance attendue sur 2025.

**Important** : la validation d'un modèle autorégressif doit se faire en **mode séquentiel**. Pour l'année de validation, on ne peut pas utiliser le vrai PM du jour précédent comme `logPM_lag1` (on triche sinon). On doit :
1. Initialiser avec le dernier PM connu de l'année de train pour chaque station.
2. Prédire jour par jour, puis utiliser la prédiction comme lag du jour suivant.

C'est exactement la même procédure qui sera utilisée pour 2025.

In [ ]:
function predict_ridge(β, X_new_std)
    return X_new_std * β
end

# --- Prédiction séquentielle pour un modèle ridge avec logPM_lag1 ---
function sequential_predict_ridge(df_target, last_known_pm;
                                   β, mu_X, sigma_X, ref_levels)
    # df_target doit être déjà trié par Date puis stationId
    df = copy(df_target)
    df.PM_pred = Vector{Float64}(undef, nrow(df))

    # Suivi du dernier PM prédit (ou observé initialement) pour chaque station
    last_pm = copy(last_known_pm)   # Dict{Int, Float64}

    # Traitement par date croissante
    dates_sorted = sort(unique(df.Date))
    for d in dates_sorted
        idx = findall(df.Date .== d)
        # Construire logPM_lag1 pour ces lignes à partir de last_pm
        for i in idx
            sid = df.stationId[i]
            prev = get(last_pm, sid, missing)
            if ismissing(prev) || prev <= 0
                # Fallback : médiane du train sur cette station
                prev = exp(mean(y_tr))  # moyenne géométrique globale
            end
            df.logPM_lag1[i] = log(prev)
        end

        # Construire la matrice de design pour ces lignes
        sub = df[idx, :]
        X_sub, _, _ = build_design_matrix(sub, use_lag=true,
                                          reference_levels=ref_levels)
        X_sub_std = standardize(X_sub, mu_X, sigma_X)
        log_pred = predict_ridge(β, X_sub_std)
        pm_pred  = exp.(log_pred)
        df.PM_pred[idx] = pm_pred

        # Mise à jour de last_pm
        for (k, i) in enumerate(idx)
            last_pm[df.stationId[i]] = pm_pred[k]
        end
    end

    return df.PM_pred
end

# --- Validation rolling-origin ---
function rolling_origin_ar_validation(df_all, years_val; use_lag=true)
    rmses = Float64[]
    df_all_sorted = sort(df_all, [:Date, :stationId])

    for y_val in years_val
        d_tr = df_all_sorted[year.(df_all_sorted.Date) .< y_val, :]
        d_va = df_all_sorted[year.(df_all_sorted.Date) .== y_val, :]

        # Nettoyer le train et enlever les valeurs manquantes de la cible
        d_tr = dropmissing(d_tr, [:PM, :logPM_lag1])
        d_va = dropmissing(d_va, :PM)

        if nrow(d_tr) == 0 || nrow(d_va) == 0
            continue
        end

        # Ajuster la ridge avec BIC sur d_tr
        X_t, _, lvls = build_design_matrix(d_tr, use_lag=use_lag)
        y_t = log.(Float64.(d_tr.PM))
        mu_l    = vec(mean(X_t[:, 2:end], dims=1))
        sigma_l = vec(std(X_t[:, 2:end], dims=1))
        sigma_l[sigma_l .== 0] .= 1.0
        X_t_std = standardize(X_t, mu_l, sigma_l)

        bic_l = [ridge_bic(X_t_std, y_t, λ)[1] for λ in lambda_grid]
        best_λ_l = lambda_grid[argmin(bic_l)]
        β_l = ridge_fit(X_t_std, y_t, best_λ_l)

        if use_lag
            # Initialiser last_pm avec le dernier PM observé en d_tr pour chaque station
            last_pm = Dict{Int, Float64}()
            for sdf in groupby(d_tr, :stationId)
                last_pm[sdf.stationId[1]] = Float64(sdf.PM[end])
            end
            preds = sequential_predict_ridge(d_va, last_pm;
                                              β=β_l, mu_X=mu_l, sigma_X=sigma_l,
                                              ref_levels=lvls)
        else
            X_v, _, _ = build_design_matrix(d_va, use_lag=false,
                                             reference_levels=lvls)
            X_v_std = standardize(X_v, mu_l, sigma_l)
            preds = exp.(predict_ridge(β_l, X_v_std))
        end

        rm = rmse(d_va.PM, preds)
        push!(rmses, rm)
        println("  Année ", y_val, " (λ=", round(best_λ_l, sigdigits=3),
                ") → RMSE = ", round(rm, digits=4))
    end
    return mean(rmses)
end

println("--- Ridge SANS lag (baseline) ---")
rmse_ridge_nolag = rolling_origin_ar_validation(train_clean, 2021:2024, use_lag=false)
println("RMSE moyen : ", round(rmse_ridge_nolag, digits=4))

println("\n--- Ridge AVEC lag (amélioration A) ---")
rmse_ridge_ar = rolling_origin_ar_validation(train_clean, 2021:2024, use_lag=true)
println("RMSE moyen : ", round(rmse_ridge_ar, digits=4))

### 4.5 Comparaison avec le modèle OLS (ancien modèle 5)

In [ ]:
# Validation rolling-origin pour les modèles OLS (linéaires avec lm)
function temporal_cv_rmse_ols(model_fn, df, years_val; use_lag=false)
    df = sort(df, [:Date, :stationId])
    rmses = Float64[]

    for y_val in years_val
        d_tr = df[year.(df.Date) .< y_val, :]
        d_va = df[year.(df.Date) .== y_val, :]

        d_tr = dropmissing(d_tr, use_lag ? [:PM, :logPM_lag1] : [:PM])
        d_va_clean = dropmissing(d_va, :PM)

        if nrow(d_tr) == 0 || nrow(d_va_clean) == 0
            continue
        end

        m = model_fn(d_tr)

        if use_lag
            # Prédiction séquentielle
            last_pm = Dict{Int, Float64}()
            for sdf in groupby(d_tr, :stationId)
                last_pm[sdf.stationId[1]] = Float64(sdf.PM[end])
            end

            d_va_sorted = sort(d_va_clean, [:Date, :stationId])
            d_va_sorted.PM_pred = Vector{Float64}(undef, nrow(d_va_sorted))
            dates_sorted = sort(unique(d_va_sorted.Date))
            for d in dates_sorted
                idx = findall(d_va_sorted.Date .== d)
                for i in idx
                    sid = d_va_sorted.stationId[i]
                    prev = get(last_pm, sid, missing)
                    d_va_sorted.logPM_lag1[i] = ismissing(prev) || prev<=0 ? mean(log.(d_tr.PM)) : log(prev)
                end
                sub = d_va_sorted[idx, :]
                try
                    log_pred = predict(m, sub)
                    pm_pred = exp.(log_pred)
                    d_va_sorted.PM_pred[idx] = pm_pred
                    for (k, i) in enumerate(idx)
                        last_pm[d_va_sorted.stationId[i]] = pm_pred[k]
                    end
                catch
                    d_va_sorted.PM_pred[idx] .= mean(d_tr.PM)
                end
            end
            push!(rmses, rmse(d_va_sorted.PM, d_va_sorted.PM_pred))
        else
            ŷ = exp.(predict(m, d_va_clean))
            push!(rmses, rmse(d_va_clean.PM, ŷ))
        end
        println("  Année ", y_val, " → RMSE = ", round(rmses[end], digits=4))
    end
    return mean(rmses)
end

println("--- OLS baseline (sans lag) ---")
rmse_ols_base = temporal_cv_rmse_ols(model_baseline, train_clean, 2021:2024, use_lag=false)
println("RMSE moyen : ", round(rmse_ols_base, digits=4))

println("\n--- OLS avec logPM_lag1 (amélioration A) ---")
rmse_ols_ar = temporal_cv_rmse_ols(model_ar, train_clean, 2021:2024, use_lag=true)
println("RMSE moyen : ", round(rmse_ols_ar, digits=4))

### 4.6 Synthèse des modèles

| Modèle | RMSE rolling-origin 2021-2024 | 
|---|---|
| OLS sans lag (ancien modèle 5) | ≈ RMSE du baseline |
| OLS avec `logPM_lag1` | ≈ RMSE AR |
| Ridge sans lag + BIC | ≈ RMSE ridge no-lag |
| Ridge avec `logPM_lag1` + BIC | **Meilleur RMSE attendu** |

Le modèle ridge + lag + BIC combine l'information la plus discriminante (PM de la veille par station), une régularisation bayésienne qui évite la multicolinéarité avec les 22+ variables explicatives, et un critère de sélection du $\lambda$. C'est ce modèle que nous retenons pour la soumission finale.

---
## 5. Prédiction finale sur 2025

### 5.1 Chargement et préparation des données de test

Nous appliquons exactement les mêmes transformations qu'à `train` : déduplication, jointure, features temporelles cycliques, amplitudes météo, indicateurs de manquance, imputation (avec les médianes du train uniquement), centrage avec les mêmes constantes $\mu_{\cdot}$.

In [ ]:
test_air = CSV.read("data/qualite-de-lair_test.csv", DataFrame)
test_meteo = CSV.read("data/meteo_test.csv", DataFrame)

# Déduplication du test (par sécurité — le test a normalement 365 dates uniques)
test_meteo = combine(groupby(test_meteo, :Date), first)

test = innerjoin(test_air, test_meteo, on=:Date, makeunique=true)
test = sort(test, [:Date, :stationId])

# Variables temporelles et cycliques
add_time_features!(test)
add_meteo_derived!(test)

# Indicateurs de manquance AVANT imputation
for var in [:NO2, :O3, :SO2]
    test[!, Symbol(string(var) * "_miss")] = Int.(ismissing.(test[!, var]))
end

# Imputation avec les médianes CALCULÉES SUR LE TRAIN
for v in vars_to_impute
    if v in Symbol.(names(test))
        imputer_par_station_mois!(test, train, v)
    end
end
imputer_par_station_mois!(test, train, :SO2)

# Variables centrées (avec les mêmes mu_* que le train)
test.NO2_c            = test.NO2 .- mu_NO2
test.temp_moy_c       = test.temp_moy .- mu_temp
test.visibilite_moy_c = test.visibilite_moy .- mu_vis
test.hum_rel_moy_c    = test.hum_rel_moy .- mu_hum
test.vent_c           = test.vitesse_vent_moy .- mu_vent

test.NO2_c_sq            = test.NO2_c .^ 2
test.temp_moy_c_sq       = test.temp_moy_c .^ 2
test.visibilite_moy_c_sq = test.visibilite_moy_c .^ 2

test.NO2_x_vent = test.NO2_c .* test.vent_c
test.temp_x_hum = test.temp_moy_c .* test.hum_rel_moy_c

# Placeholder pour logPM_lag1 (sera rempli séquentiellement)
test.logPM_lag1 = Vector{Union{Missing, Float64}}(missing, nrow(test))

println("Dimensions du jeu de test : ", size(test))
println("Valeurs manquantes restantes dans les colonnes utilisées :")
cols_to_check = [:NO2_c, :visibilite_moy_c, :vent_c, :temp_moy_c, :hum_rel_moy_c,
                  :pression_moy, :O3, :pluie, :neige, :neige_au_sol,
                  :sin_doy, :cos_doy, :sin_doy2, :cos_doy2,
                  :NO2_x_vent, :temp_x_hum,
                  :NO2_miss, :O3_miss, :SO2_miss, :station_str]
for c in cols_to_check
    n_miss = sum(ismissing.(test[!, c]))
    n_miss > 0 && println("  ", c, " : ", n_miss)
end

### 5.2 Initialisation du lag à partir du dernier PM connu de 2024

Pour chaque station, on récupère le PM du **dernier jour observé en 2024**, qui sert de valeur initiale pour `logPM_lag1` le 1er janvier 2025.

In [ ]:
# AMÉLIORATION B : préparation de l'initialisation pour la prédiction séquentielle
train_end_2024 = dropmissing(train[year.(train.Date) .== 2024, :], :PM)
sort!(train_end_2024, [:stationId, :Date])

last_known_pm = Dict{Int, Float64}()
for sdf in groupby(train_end_2024, :stationId)
    last_known_pm[sdf.stationId[1]] = Float64(sdf.PM[end])
end

# Pour les stations qui n'auraient pas eu de mesure en 2024 (peu probable),
# on utilise la médiane globale sur le train
median_pm_train = median(skipmissing(train.PM))
for sid in unique(test.stationId)
    if !haskey(last_known_pm, sid)
        last_known_pm[sid] = median_pm_train
    end
end

println("PM initial pour chaque station (dernière valeur 2024) :")
for (k, v) in sort(collect(last_known_pm))
    println("  Station ", k, " : ", round(v, digits=2))
end

### 5.3 Entraînement final et prédiction séquentielle

On ré-entraîne le modèle ridge sur toutes les données `train_clean` (2007-2024, après retrait des outliers), puis on génère les prédictions 2025 jour par jour, en propageant le lag.

In [ ]:
# Données d'entraînement final : train_clean avec lag disponible
train_final = dropmissing(train_clean, [:PM, :logPM_lag1])
X_full, full_var_names, full_ref_levels = build_design_matrix(train_final, use_lag=true)
y_full = log.(Float64.(train_final.PM))

mu_full    = vec(mean(X_full[:, 2:end], dims=1))
sigma_full = vec(std(X_full[:, 2:end], dims=1))
sigma_full[sigma_full .== 0] .= 1.0
X_full_std = standardize(X_full, mu_full, sigma_full)

# BIC pour choisir λ sur l'ensemble du train
bic_full = [ridge_bic(X_full_std, y_full, λ)[1] for λ in lambda_grid]
best_lambda_final = lambda_grid[argmin(bic_full)]
β_final = ridge_fit(X_full_std, y_full, best_lambda_final)

println("Modèle final (ridge + lag + BIC) :")
println("  λ optimal = ", round(best_lambda_final, sigdigits=4))
println("  Nombre de coefficients = ", length(β_final))

# Prédictions séquentielles sur 2025
preds_2025 = sequential_predict_ridge(test, last_known_pm;
                                       β=β_final, mu_X=mu_full,
                                       sigma_X=sigma_full,
                                       ref_levels=full_ref_levels)

println("\nPrédictions 2025 générées : ", length(preds_2025), " valeurs")
println("Statistiques des prédictions :")
println("  Moyenne : ", round(mean(preds_2025), digits=2))
println("  Écart-type : ", round(std(preds_2025), digits=2))
println("  Min / Max : ", round(minimum(preds_2025), digits=2), " / ",
        round(maximum(preds_2025), digits=2))

### 5.4 Préparation du fichier de soumission

Nous préparons le fichier de soumission pour avoir les colonnes demandés pour la compétition sur Kaggle.

In [ ]:
function createID(date::Vector{<:Date}, station::Vector{<:Any})
    n = length(date)
    ID = Array{Tuple}(undef, n)
    for i in 1:n
        ID[i] = (date[i], station[i])
    end
    return ID
end

# IMPORTANT : test est trié par [Date, stationId], preds_2025 suit le même ordre
benchmarkID = createID(test.Date, test.stationId)
benchmark_predictions = DataFrame(ID = benchmarkID, PM = preds_2025)

println("Aperçu du fichier de soumission :")
println(first(benchmark_predictions, 5))
println("\nNombre de prédictions : ", nrow(benchmark_predictions),
        " (attendu : 2555)")

CSV.write("benchmark_predictions.csv", benchmark_predictions)
println("\nFichier écrit : benchmark_predictions.csv")

# 6. Conclusion et améliorations

### Pistes avortées

Nous avons exploré une méthode de sélection de variables pas-à-pas ascendante, qui s'est révélée peu performante, probablement en raison de la forte multicolinéarité entre les variables explicatives. En présence de telle corrélation, la sélection pas-à-pas tend à être instable et à exclure des variables pourtant pertinentes. Il aurait été préférable de se tourner plus rapidement vers une méthode de régularisation comme Ridge, qui gère naturellement ce problème.



### Conclusion

En conclusion, ce travail souligne l'importance d'une préparation rigoureuse des données et d'une validation croisée robuste pour produire des prédictions fiables dans un contexte de santé publique. Malgré les défis rencontrés, notamment en ce qui concerne les données manquantes et la multicolinéarité, nous avons pu identifier des variables explicatives pertinentes et construire un modèle qui a permis d'obtenir des prédictions raisonnables pour l'année 2025. L'utilisation d'une régression ridge avec sélection de $\lambda$ par BIC a permis de gérer efficacement la complexité du modèle.

### Améliorations possibles

Les résultats obtenus sont encourageants, cependant plusieurs pistes d'amélioration sont envisageables pour affiner davantage les prédictions :
- Modèles non linéaires : les relations entre PM et les variables explicatives pourraient être non linéaires. Il aurait peut-être été intéressant de modéliser avec des nouvelles méthodes comme les forêts aléatoires ou les réseaux de neurones, qui captent mieux les interactions complexes.
- Variables en lag : nous avons utilisé uniquement le PM de la veille, mais il serait pertinent d'explorer des lags plus longs (t-2, t-3, etc.) pour capturer une mémoire plus longue de la pollution.
- Effets d'interaction : nous avons introduit quelques interactions, mais il serait intéressant d'explorer systématiquement les interactions entre les variables météorologiques et les polluants.
- Omission de $SO_2$ : Nous aurions pu créé un modèle spécifique pour prédire plus précisément $SO_2$ à partir des autres variables, puis utiliser ces prédictions comme variable explicative dans le modèle de PM.



